In [1]:
import joblib
import numpy as np
import pandas as pd
import librosa
import os

In [3]:
model_queen_presence=joblib.load("queen_presence_model.pkl")
model_anomaly = joblib.load("bee_sound_anomaly_model.pkl")
model_package = joblib.load('queen_acceptance_model_package.pkl')

# Access the components
model = model_package['model']
scaler = model_package['scaler']
label_encoder = model_package['label_encoder']

In [9]:
audio_path = r"D:\AIDS\3rd year\sem2\Bee Hive Condition Projecct\archive\sound_files\split_audio\2022-06-13--18-08-05_1__segment0_part1.wav"
hive_temp = 25
hive_humidity = 28
weather_temp = 34
weather_humidity = 27
# audio_path = input("Enter the full path of the audio clip (.wav): ").strip('"')
# hive_temp = float(input("Enter hive temperature: "))
# hive_humidity = float(input("Enter hive humidity: "))
# weather_temp = float(input("Enter weather temperature: "))
# weather_humidity = float(input("Enter weather humidity: "))

In [10]:
def extract_audio_features(file_path, n_mfcc=13):
    y, sr = librosa.load(file_path, sr=22050)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfccs_mean = np.mean(mfccs.T, axis=0)
    return mfccs_mean

mfcc_features = extract_audio_features(audio_path)
mfcc_names = [f"mfcc_{i+1}" for i in range(len(mfcc_features))]
all_features = np.concatenate(([hive_temp, hive_humidity, weather_temp, weather_humidity], mfcc_features))
all_feature_names = ["hive temp", "hive humidity", "weather temp", "weather humidity"] + mfcc_names
input_df = pd.DataFrame([all_features], columns=all_feature_names)

In [11]:
pred_qp = model_queen_presence.predict(input_df)[0]

# === OUTPUT ===
print("\n🧠 Predictions:")
print(f"🐝 Queen Presence: {'Yes' if pred_qp == 1 else 'No'}")


🧠 Predictions:
🐝 Queen Presence: Yes


In [12]:
if pred_qp==1:
    input_scaled = scaler.transform(input_df)
    prediction = model.predict(input_scaled)
    predicted_label = label_encoder.inverse_transform(prediction)

    print("Predicted Queen Acceptance:", predicted_label[0])

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=0.5, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.5
[LightGBM] [Warning] lambda_l2 is set=0.5, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.5
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
Predicted Queen Acceptance: 2


In [13]:
pred_anomaly = model_anomaly.predict(input_df)[0] 
print(f"🚨 Anomaly Detected in Sound: {'Yes' if pred_anomaly == -1 else 'No'}")


🚨 Anomaly Detected in Sound: Yes
